## <span style="color:#0b486b">Part 3: Convolutional Neural Networks and Image Classification</span>

**<div style="text-align: right"><span style="color:red">[Total marks for this part: 45 points]</span></div>**

The third part of this assignment is to demonstrate your basis knowledge in deep learning that you have acquired from the lectures and tutorials materials. Most of the contents in this assignment are drawn from **the tutorials covered from weeks 3 to 6**. Going through these materials before attempting this assignment is highly recommended.

**The dataset used for this part is a specific dataset for this unit consisting of approximately $10,000$ images of $20$ classes of Animals, each of which has approximately 500 images. You can download the dataset at [download here](https://drive.google.com/file/d/1aEkxNWaD02Z8ZNvZzeMefUoY97C-3wTG/view?usp=drive_link) if you want to do your assignment on your machine.**


In [ ]:
import os
import requests
import tarfile
import time
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torchvision.models as models
import torch.nn as nn
import torch
import PIL.Image
import pathlib
from torchsummary import summary
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("No GPU found, using CPU instead")

# check if CUDA is available
train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(1234)

In [ ]:
data_dir = r"C:\Users\User\Documents\5215\FIT5215_Dataset"

# We resize the images to [3,64,64]
transform = transforms.Compose([transforms.Resize((64,64)),  #resises the image so it can be perfect for our model.
                                      transforms.RandomHorizontalFlip(), # FLips the image w.r.t horizontal axis
                                      #transforms.RandomRotation(4),     #Rotates the image to a specified angel
                                      #transforms.RandomAffine(0, shear=10, scale=(0.8,1.2)), #Performs actions like zooms, change shear angles.
                                      transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), # Set the color params
                                      transforms.ToTensor(), # convert the image to tensor so that it can work with torch
                                      transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # Normalize the images, each R,G,B value is normalized with mean=0.5 and std=0.5
                                      ])


# Load the dataset using torchvision.datasets.ImageFolder and apply transformations
dataset = datasets.ImageFolder(data_dir, transform=transform)

# Split the dataset into training and validation sets
train_size = int(0.9 * len(dataset))
valid_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, valid_size])

# Example of DataLoader creation for training and validation
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Number of instance in train_set: %s" % len(train_dataset))
print("Number of instance in val_set: %s" % len(val_dataset))

In [ ]:
class_names = ['bird', 'bottle', 'bread', 'butterfly', 'cake', 'cat', 'chicken', 'cow', 'dog', 'duck',
                  'elephant', 'fish', 'handgun', 'horse', 'lion', 'lipstick', 'seal', 'snake', 'spider', 'vase']

In [ ]:
# obtain one batch of training images
dataiter = iter(train_loader)
images, labels = next(dataiter)
images = images.numpy() # convert images to numpy for display

In [ ]:
import math

def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    plt.imshow(np.transpose(img, (1, 2, 0)))  # convert from Tensor image

def visualize_data(images, categories, images_per_row = 8):
    class_names = ['bird', 'bottle', 'bread', 'butterfly', 'cake', 'cat', 'chicken', 'cow', 'dog', 'duck',
                  'elephant', 'fish', 'handgun', 'horse', 'lion', 'lipstick', 'seal', 'snake', 'spider', 'vase']
    n_images = len(images)
    n_rows = math.ceil(float(n_images)/images_per_row)
    fig = plt.figure(figsize=(1.5*images_per_row, 1.5*n_rows))
    fig.patch.set_facecolor('white')
    for i in range(n_images):
        plt.subplot(n_rows, images_per_row, i+1)
        plt.xticks([])
        plt.yticks([])
        imshow(images[i])
        class_index = categories[i]
        plt.xlabel(class_names[class_index])
    plt.show()

In [ ]:
visualize_data(images, labels)

**For questions 3.1 to 3.7, you'll need to write your own model in a way that makes it easy for you to experiment with different architectures and parameters. The goal is to be able to pass the parameters to initialize a new instance of `YourModel` to build different network architectures with different parameters. Below are descriptions of some parameters for `YourModel`:**

1. `Block confirguration`: Our network consists of many blocks. Each block has the pattern `[conv, batch norm, activation, conv, batch norm, activation, max pool, dropout]`. All convolutional layers have filter size $(3, 3)$, strides $(1, 1)$ and padding = 1, and all max pool layers have strides $(2, 2)$, kernel size $2$, and padding = 0. The network will consists of a few blocks before applying a linear layer to output the logits for the softmax layer.

2. `list_feature_maps`: the number of feature maps in the blocks of the network. For example, if `list_feature_maps = [16, 32, 64]`, our network has two blocks with the input_channels or number of feature maps are `16, 32`, and `64` respectively.
3. `drop_rate`: the keep probability for dropout. Setting `drop_rate` to $0.0$ means not using dropout.
4. `batch_norm`: the batch normalization function is used or not. Setting `batch_norm` to `false` means not using batch normalization.
5. `use_skip`: the skip connection is used in the blocks or not. Setting this to `true` means that we use `1x1` Conv2D with `strides=2` for the skip connection.
6. At the end, you need to apply `global average pooling (GAP)` (`AdaptiveAvgPool2d((1, 1))`) to flatten the 3D output tensor before defining the output linear layer for predicting the labels.



[Embedded assignment diagram omitted from portfolio export]

<span style="color:red">**Question 3.1:**</span> **You need to implement the aforementioned CNN**.

**First, you need to implement the block of our CNN in the class `YourBlock`. You can ignore `use_skip` and `skip connection` for simplicity. However, you cannot earn full marks for this question.**

<div style="text-align: right"> <span style="color:red">[6 points]</span> </div>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================================================
# Task 3.1: Define YourBlock
# =========================================================
class YourBlock(nn.Module):
    def __init__(self, in_channels, out_channels, drop_rate=0.2, batch_norm=True, use_skip=True):
        super(YourBlock, self).__init__()
        self.use_skip = use_skip

        layers = []
        # Conv → BN → ReLU
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
        if batch_norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))

        # Conv → BN → ReLU
        layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1))
        if batch_norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))

        # MaxPool → Dropout
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2, padding=0))
        layers.append(nn.Dropout2d(drop_rate))

        self.block = nn.Sequential(*layers)

        # Skip connection (1x1 Conv with stride=2 to match downsampling)
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=2) if use_skip else None

    def forward(self, x):
        identity = x
        out = self.block(x)
        if self.use_skip:
            identity = self.skip(identity)
        out += identity
        return out






**Second, you need to use the above `YourBlock` to implement the class `YourCNN`**.

<div style="text-align: right"> <span style="color:red">[6 points]</span> </div>

In [ ]:
class YourCNN(nn.Module):
    def __init__(self, list_feature_maps=[16, 32, 64], drop_rate=0.2, batch_norm=True, use_skip=True, num_classes=20):
        super(YourCNN, self).__init__()
        layers = []
        for i, out_channels in enumerate(list_feature_maps):
            in_channels = 3 if i == 0 else list_feature_maps[i-1]
            layers.append(YourBlock(in_channels, out_channels, drop_rate, batch_norm, use_skip))

        self.features = nn.ModuleList(layers)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(list_feature_maps[-1], num_classes)

    def forward(self, x):
        for block in self.features:
            x = block(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)  # flatten except batch dim
        x = self.classifier(x)
        return x



**We declare `my_cnn` from `YourCNN` as follows**.

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
my_cnn = YourCNN(list_feature_maps = [16, 32, 64], use_skip = True)
my_cnn = my_cnn.to(device)
print(my_cnn)

**We declare the optimizer and the loss function.**

In [ ]:
# Loss and optimizer
learning_rate = 0.001
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(my_cnn.parameters(), lr=learning_rate)

**Here are the codes to compute the loss and accuracy.**

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def compute_loss(model, loss_fn, loader):
  loss = 0
  # Set model to eval mode for inference
  model.eval()
  with torch.no_grad():  # No need to track gradients for validation
    for (batchX, batchY) in loader:
      # Move data to the same device as the model
      batchX, batchY = batchX.to(device).type(torch.float32), batchY.to(device).type(torch.long)
      loss += loss_fn(model(batchX), batchY)
  # Set model back to train mode
  model.train()
  return float(loss)/len(loader)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def compute_acc(model, loader):
    correct = 0
    totals = 0
    # Set model to eval mode for inference
    model.eval()
    for (batchX, batchY) in loader:
        # Move batchX and batchY to the same device as the model
        batchX, batchY = batchX.to(device).type(torch.float32), batchY.to(device)
        outputs = model(batchX)  # feed batch to the model
        totals += batchY.size(0)  # accumulate totals with the current batch size
        predicted = torch.argmax(outputs.data, 1)  # get the predicted class
        # Move batchY to the same device as predicted for comparison
        correct += (predicted == batchY).sum().item()
    return correct / totals

**Here is the code to train our model.**

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def fit(model= None, train_loader = None, valid_loader= None, optimizer = None,
        num_epochs = 50, verbose = True, seed= 1234):
  torch.manual_seed(seed)
  # Move the model to the device before initializing the optimizer
  model.to(device) # Move the model to the GPU

  if optimizer == None:
    optim = torch.optim.Adam(model.parameters(), lr = 0.001) # Now initialize optimizer with model on GPU
  else:
    optim = optimizer
  history = dict()
  history['val_loss'] = list()
  history['val_acc'] = list()
  history['train_loss'] = list()
  history['train_acc'] = list()

  for epoch in range(num_epochs):
    model.train()
    for (X, y) in train_loader:
      # Move input data to the same device as the model
      X,y = X.to(device), y.to(device)
      # Forward pass
      outputs = model(X.type(torch.float32)) # X is already on the correct device
      loss = loss_fn(outputs, y.type(torch.long))
      # Backward and optimize
      optim.zero_grad()
      loss.backward()
      optim.step()
      #losses and accuracies for epoch
    val_loss = compute_loss(model, loss_fn, valid_loader)
    val_acc = compute_acc(model, valid_loader)
    train_loss = compute_loss(model, loss_fn, train_loader)
    train_acc = compute_acc(model, train_loader)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    if not verbose: #verbose = True means we do show the training information during training
      print(f"Epoch {epoch+1}/{num_epochs}")
      print(f"train loss= {train_loss:.4f} - train acc= {train_acc*100:.2f}% - valid loss= {val_loss:.4f} - valid acc= {val_acc*100:.2f}%")
  return history

In [ ]:
history = fit(model= my_cnn, train_loader=train_loader, valid_loader = val_loader, optimizer = optimizer, num_epochs= 10, verbose = False)

<span style="color:red">**Question 3.2:**</span> **Now, let us tune the number of blocks, $use\_skip \in \{true,false\}$ and $learning\_rate \in \{0.001, 0.0005\}$. Write your code for this tuning and report the result of the best model on the testing set. Note that you need to show your code for tuning and evaluating on the test set to earn the full marks. During tuning, you can set the instance variable `verbose` of your model to `True` for not showing the training details of each epoch.**

Note that for this question, depending on your computational resource, you can choose `list_feature_maps= [32, 64]` or `list_feature_maps= [16, 32, 64]`.

<div style="text-align: right"> <span style="color:red">[3 points]</span> </div>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import copy

# Data preparation: train / val / test
data_dir = r"C:\Users\User\Documents\5215\FIT5215_Dataset"

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = datasets.ImageFolder(data_dir, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# Basic building block for CNN
class YourBlock(nn.Module):
    def __init__(self, in_channels, out_channels, drop_rate=0.2, batch_norm=True, use_skip=True):
        super(YourBlock, self).__init__()
        self.use_skip = use_skip
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels) if batch_norm else nn.Identity(),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels) if batch_norm else nn.Identity()
        ]
        self.block = nn.Sequential(*layers)
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if use_skip and in_channels != out_channels else None
        self.dropout = nn.Dropout2d(drop_rate)

    def forward(self, x):
        identity = x
        out = self.block(x)
        if self.use_skip:
            if self.skip is not None:
                identity = self.skip(identity)
            if identity.shape == out.shape:
                out += identity
        out = F.relu(out)
        return self.dropout(out)


# CNN model definition
class YourCNN(nn.Module):
    def __init__(self, list_feature_maps=[16,32,64], drop_rate=0.2, batch_norm=True, use_skip=True, num_classes=20):
        super(YourCNN, self).__init__()
        layers = []
        for i, out_channels in enumerate(list_feature_maps):
            in_channels = 3 if i==0 else list_feature_maps[i-1]
            layers.append(YourBlock(in_channels, out_channels, drop_rate, batch_norm, use_skip))
        self.features = nn.ModuleList(layers)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Linear(list_feature_maps[-1], num_classes)

    def forward(self, x):
        for block in self.features:
            x = block(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# Training and validation function
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        running_corrects, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()
            running_corrects += torch.sum(preds == labels.data)
            total += labels.size(0)
        train_acc = running_corrects.double() / total

        # Validation
        model.eval()
        val_corrects, total_val = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                total_val += labels.size(0)
        val_acc = val_corrects.double() / total_val

        print(f"Epoch {epoch+1}/{num_epochs} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model, best_val_acc


# Hyperparameter tuning function
def hyperparameter_tuning():
    configs = [
        ([32,64], True, 0.001),
        ([32,64], True, 0.0005),
        ([32,64], False, 0.001),
        ([32,64], False, 0.0005),
        ([16,32,64], True, 0.001),
        ([16,32,64], True, 0.0005),
        ([16,32,64], False, 0.001),
        ([16,32,64], False, 0.0005),
    ]
    best_val_acc, best_config, best_model = 0.0, None, None
    for i, (fm, skip, lr) in enumerate(configs, 1):
        print(f"\nConfig {i}/{len(configs)}: feature_maps={fm}, use_skip={skip}, lr={lr}")
        model = YourCNN(list_feature_maps=fm, use_skip=skip)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        trained_model, val_acc = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5)
        print(f"Validation Accuracy: {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc, best_config, best_model = val_acc, (fm, skip, lr), copy.deepcopy(trained_model)
    test_acc = evaluate_model(best_model, test_loader)
    print(f"\nBest Config: {best_config}, Best Val Acc: {best_val_acc:.4f}, Final Test Acc: {test_acc:.4f}")
    return best_model, best_config


# Test evaluation function
def evaluate_model(model, test_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    corrects, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
            total += labels.size(0)
    return corrects.double() / total


# Run tuning
best_model, best_config = hyperparameter_tuning()


<span style="color:red">**Question 3.3:**</span> **Exploring Data Mixup Technique for Improving Generalization Ability.**
   
<div style="text-align: right"> <span style="color:red">[4 points]</span> </div>

Data mixup is another super-simple technique used to boost the generalization ability of deep learning models. You need to incoroporate data mixup technique to the above deep learning model and experiment its performance. There are some papers and documents for data mixup as follows:
- Main paper for data mixup [link for main paper](https://openreview.net/pdf?id=r1Ddp1-Rb) and a good article [article link](https://www.inference.vc/mixup-data-dependent-data-augmentation/).

You need to extend your model developed above, train a model using data mixup, and write your observations and comments about the result.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import copy

# Mixup utility functions

def mixup_data(x, y, alpha=0.2, use_cuda=True):
    """Return mixed inputs, pairs of targets, and lambda"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device if use_cuda else "cpu")
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixup loss"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)



# Training functions (with and without mixup)

def train_model_with_mixup(model, train_loader, val_loader, criterion, optimizer,
                           num_epochs=10, alpha=0.2, verbose=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        if verbose:
            print(f"Epoch {epoch+1}/{num_epochs}\n{'-'*10}")

        model.train()
        running_loss, running_corrects, total_samples = 0.0, 0.0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha, device.type=="cuda")

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)

            _, preds = torch.max(outputs, 1)
            correct_a = torch.sum(preds == targets_a).item()
            correct_b = torch.sum(preds == targets_b).item()
            running_corrects += lam * correct_a + (1 - lam) * correct_b

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            total_samples += inputs.size(0)

        train_acc = running_corrects / total_samples

        # Validation
        model.eval()
        val_corrects, total_val = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels).item()
                total_val += labels.size(0)

        val_acc = val_corrects / total_val

        if verbose:
            print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model, best_val_acc


def train_model_without_mixup(model, train_loader, val_loader, criterion, optimizer,
                              num_epochs=10, verbose=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        if verbose:
            print(f"Epoch {epoch+1}/{num_epochs}\n{'-'*10}")

        model.train()
        running_corrects, total_samples = 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()

            running_corrects += torch.sum(preds == labels).item()
            total_samples += labels.size(0)

        train_acc = running_corrects / total_samples

        # Validation
        model.eval()
        val_corrects, total_val = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels).item()
                total_val += labels.size(0)

        val_acc = val_corrects / total_val

        if verbose:
            print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model, best_val_acc



# Test evaluation function

def evaluate_model(model, test_loader, verbose=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    corrects, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels).item()
            total += labels.size(0)
    acc = corrects / total
    if verbose:
        print(f"Test Accuracy: {acc:.4f}")
    return acc



# Experiment: Mixup vs Baseline

def compare_mixup_performance():
    print("COMPARING PERFORMANCE: MIXUP vs BASELINE\n" + "="*50)

    # Use the best config from 3.2
    best_feature_maps = [16, 32, 64]
    best_lr = 0.001

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    criterion = nn.CrossEntropyLoss()
    results = {}

    for alpha in [0.0, 0.2, 0.4]:
        model = YourCNN(list_feature_maps=best_feature_maps,
                        drop_rate=0.2, batch_norm=True, use_skip=True)
        optimizer = optim.Adam(model.parameters(), lr=best_lr)

        if alpha == 0.0:
            trained_model, val_acc = train_model_without_mixup(
                model, train_loader, val_loader, criterion, optimizer,
                num_epochs=15, verbose=True)
        else:
            trained_model, val_acc = train_model_with_mixup(
                model, train_loader, val_loader, criterion, optimizer,
                num_epochs=15, alpha=alpha, verbose=True)

        test_acc = evaluate_model(trained_model, test_loader, verbose=False)
        results[alpha] = {"val_acc": val_acc, "test_acc": test_acc}
        print(f"[α={alpha}] Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}")

    return results


# Run experiment

mixup_results = compare_mixup_performance()

print("\n" + "="*60)
print("SUMMARY OF MIXUP EXPERIMENT")
print("="*60)
for alpha, res in mixup_results.items():
    method = "Baseline (No Mixup)" if alpha == 0.0 else f"Mixup (α={alpha})"
    print(f"{method:<20} | Val Acc: {res['val_acc']:.4f} | Test Acc: {res['test_acc']:.4f}")

best_alpha = max(mixup_results, key=lambda k: mixup_results[k]['test_acc'])
best_method = "Baseline (No Mixup)" if best_alpha == 0.0 else f"Mixup (α={best_alpha})"
print(f"\nBest Method: {best_method} with Test Accuracy {mixup_results[best_alpha]['test_acc']:.4f}")


<span style="color:red">**Question 3.4:**</span> **Exploring CutMix Technique for Improving Generalization Ability.**
   
<div style="text-align: right"> <span style="color:red">[4 points]</span> </div>

CutMix is another super-simple technique used to boost the generalization ability of deep learning models. You need to incoroporate data CutMix technique to the above deep learning model and experiment its performance. There are some papers and documents for data mixup as follows:
- Main paper for Cutmix [link for main paper](https://arxiv.org/pdf/1905.04899v2) and a good article [article link](https://shorturl.at/TNwjv).

You need to extend your model developed above, train a model using data CutMix, and write your observations and comments about the result.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import copy


# CutMix helpers

def rand_bbox(size, lam):
    """Generate random bounding box"""
    W, H = size[2], size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)

    cx, cy = np.random.randint(W), np.random.randint(H)
    bbx1, bby1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    bbx2, bby2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2

def cutmix_data(x, y, alpha=1.0, use_cuda=True):
    """Return CutMix-ed samples"""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).cuda() if use_cuda else torch.randperm(batch_size)

    y_a, y_b = y, y[index]
    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bby1:bby2, bbx1:bbx2] = x[index, :, bby1:bby2, bbx1:bbx2]
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))  # adjust lam
    return x, y_a, y_b, lam

def cutmix_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# Training functions

def train_model_without_mixup(model, train_loader, val_loader, criterion, optimizer,
                              num_epochs=10, verbose=True):
    """Baseline training"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    best_val_acc, best_model_wts = 0.0, copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        running_corrects, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()
            running_corrects += torch.sum(preds == labels).item()
            total += labels.size(0)
        train_acc = running_corrects / total

        # Validation
        model.eval()
        val_corrects, total_val = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels).item()
                total_val += labels.size(0)
        val_acc = val_corrects / total_val

        if verbose:
            print(f"Epoch {epoch+1}/{num_epochs} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc, best_model_wts = val_acc, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model, best_val_acc

def train_model_with_mixup(model, train_loader, val_loader, criterion, optimizer,
                           num_epochs=10, alpha=0.2, verbose=True):
    """Training with Mixup"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    best_val_acc, best_model_wts = 0.0, copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        running_corrects, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            lam = np.random.beta(alpha, alpha)
            index = torch.randperm(inputs.size(0)).to(device)
            mixed_inputs = lam * inputs + (1 - lam) * inputs[index, :]
            targets_a, targets_b = labels, labels[index]

            optimizer.zero_grad()
            outputs = model(mixed_inputs)
            loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()
            running_corrects += lam * torch.sum(preds == targets_a).item() + (1 - lam) * torch.sum(preds == targets_b).item()
            total += labels.size(0)
        train_acc = running_corrects / total

        # Validation
        model.eval()
        val_corrects, total_val = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels).item()
                total_val += labels.size(0)
        val_acc = val_corrects / total_val

        if verbose:
            print(f"Epoch {epoch+1}/{num_epochs} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc, best_model_wts = val_acc, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model, best_val_acc

def train_model_with_cutmix(model, train_loader, val_loader, criterion, optimizer,
                            num_epochs=10, alpha=1.0, cutmix_prob=0.5, verbose=True):
    """Training with CutMix"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    best_val_acc, best_model_wts = 0.0, copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        running_corrects, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            r = np.random.rand()
            if r < cutmix_prob:
                inputs, targets_a, targets_b, lam = cutmix_data(inputs, labels, alpha, device.type == "cuda")
                outputs = model(inputs)
                loss = cutmix_criterion(criterion, outputs, targets_a, targets_b, lam)
                _, preds = torch.max(outputs, 1)
                running_corrects += lam * torch.sum(preds == targets_a).item() + (1 - lam) * torch.sum(preds == targets_b).item()
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                running_corrects += torch.sum(preds == labels).item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += labels.size(0)
        train_acc = running_corrects / total

        # Validation
        model.eval()
        val_corrects, total_val = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels).item()
                total_val += labels.size(0)
        val_acc = val_corrects / total_val

        if verbose:
            print(f"Epoch {epoch+1}/{num_epochs} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc, best_model_wts = val_acc, copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model, best_val_acc


# Evaluation

def evaluate_model(model, test_loader, verbose=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    corrects, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels).item()
            total += labels.size(0)
    acc = corrects / total
    if verbose:
        print(f"Test Accuracy: {acc:.4f}")
    return acc


# Comprehensive Comparison

def compare_all_techniques():
    best_feature_maps, best_lr = [16, 32, 64], 0.001
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    experiments = [
        {'name': 'Baseline', 'method': 'baseline'},
        {'name': 'Mixup (α=0.2)', 'method': 'mixup', 'alpha': 0.2},
        {'name': 'Mixup (α=0.4)', 'method': 'mixup', 'alpha': 0.4},
        {'name': 'CutMix (α=1.0, p=0.5)', 'method': 'cutmix', 'alpha': 1.0, 'prob': 0.5},
        {'name': 'CutMix (α=1.0, p=1.0)', 'method': 'cutmix', 'alpha': 1.0, 'prob': 1.0},
    ]

    results = {}
    for exp in experiments:
        print(f"\n=== {exp['name']} ===")
        model = YourCNN(list_feature_maps=best_feature_maps, drop_rate=0.2, batch_norm=True, use_skip=True)
        criterion, optimizer = nn.CrossEntropyLoss(), optim.Adam(model.parameters(), lr=best_lr)

        if exp['method'] == 'baseline':
            trained_model, val_acc = train_model_without_mixup(model, train_loader, val_loader, criterion, optimizer, num_epochs=15)
        elif exp['method'] == 'mixup':
            trained_model, val_acc = train_model_with_mixup(model, train_loader, val_loader, criterion, optimizer, num_epochs=15, alpha=exp['alpha'])
        elif exp['method'] == 'cutmix':
            trained_model, val_acc = train_model_with_cutmix(model, train_loader, val_loader, criterion, optimizer, num_epochs=15, alpha=exp['alpha'], cutmix_prob=exp['prob'])

        test_acc = evaluate_model(trained_model, test_loader, verbose=False)
        results[exp['name']] = {'val_accuracy': val_acc, 'test_accuracy': test_acc}
        print(f"Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}")

    print("\nSUMMARY")
    for name, res in results.items():
        print(f"{name:<20} | Val: {res['val_accuracy']:.4f} | Test: {res['test_accuracy']:.4f}")
    return results


# Run

all_results = compare_all_techniques()


<span style="color:red">**Question 3.5:**</span> Implement the **one-versus-all (OVA)** loss. The details are as follows:
- You need to apply `the sigmoid activation function` to logits $h = [h_1, h_2,...,h_M]$ instead of `the softmax activation` function as usual to obtain $p = [p_1, p_2,...,p_M]$, meaning that $p_i = sigmoid(h_i), i=1,...,M$. Note that $M$ is the number of classes.
- Given a data example $x$ with the ground-truth label $y$, the idea is to maximize the likelihood $p_y$ and to minimize the likelihoods $p_i, i \neq y$. Therefore, the objective function is to find the model parameters to
  - $\max\left\{ \log p_{y}+\sum_{i\neq y}\log(1-p_{i})\right\}$ or equivalently $\min\left\{ -\log p_{y}-\sum_{i\neq y}\log(1-p_{i})\right\}$.
  - For example, if $M=3$ and $y=2$, you need to minimize $\min\left\{ -\log(1-p_{1})-\log p_{2}-\log(1-p_{3})\right\}$.

Compare the model trained with the OVA loss and the same model trained with the standard cross-entropy loss.

<div style="text-align: right"> <span style="color:red">[4 points]</span> </div>



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import copy

# Basic CNN Block

class YourBlock(nn.Module):
    def __init__(self, in_channels, out_channels, drop_rate=0.2, batch_norm=True, use_skip=True):
        super(YourBlock, self).__init__()
        self.use_skip = use_skip
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        ]
        if batch_norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1))
        if batch_norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        layers.append(nn.Dropout2d(p=drop_rate))
        self.block = nn.Sequential(*layers)
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=2) \
            if use_skip and in_channels != out_channels else None

    def forward(self, x):
        identity = x
        out = self.block(x)
        if self.use_skip and self.skip is not None:
            identity = self.skip(identity)
            if identity.shape == out.shape:
                out += identity
        return out


# CNN Models

class YourCNN(nn.Module):
    def __init__(self, list_feature_maps=[16, 32, 64], drop_rate=0.2, batch_norm=True, use_skip=True, num_classes=20):
        super(YourCNN, self).__init__()
        layers = []
        in_channels = 3
        for out_channels in list_feature_maps:
            layers.append(YourBlock(in_channels, out_channels, drop_rate, batch_norm, use_skip))
            in_channels = out_channels
        self.features = nn.Sequential(*layers)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(list_feature_maps[-1], num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

class YourCNN_OVA(YourCNN):
    def forward(self, x):
        return super().forward(x)  # logits, sigmoid applied in loss


# OVA Loss

class OVALoss(nn.Module):
    def __init__(self):
        super(OVALoss, self).__init__()

    def forward(self, logits, targets):
        batch_size, num_classes = logits.shape
        probs = torch.sigmoid(logits)
        targets_onehot = F.one_hot(targets, num_classes=num_classes).float()
        pos_loss = -torch.log(probs + 1e-8) * targets_onehot
        neg_loss = -torch.log(1 - probs + 1e-8) * (1 - targets_onehot)
        total_loss = pos_loss + neg_loss
        return total_loss.sum(dim=1).mean()


# Training Function

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=15, use_sigmoid=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        correct, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, labels)
            if use_sigmoid:
                preds = torch.argmax(torch.sigmoid(logits), 1)
            else:
                preds = torch.argmax(logits, 1)
            loss.backward()
            optimizer.step()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        train_acc = correct / total

        # validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                logits = model(inputs)
                if use_sigmoid:
                    preds = torch.argmax(torch.sigmoid(logits), 1)
                else:
                    preds = torch.argmax(logits, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total
        print(f"Epoch {epoch+1}/{num_epochs} - Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model, best_val_acc


# Evaluation

def evaluate_model(model, test_loader, use_sigmoid=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            if use_sigmoid:
                preds = torch.argmax(torch.sigmoid(logits), 1)
            else:
                preds = torch.argmax(logits, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


# Compare CE vs OVA

def compare_ova_vs_crossentropy():
    best_feature_maps = [16, 32, 64]
    lr = 0.001
    epochs = 15
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # CE
    print("\n=== CrossEntropy ===")
    model_ce = YourCNN(best_feature_maps)
    opt_ce = optim.Adam(model_ce.parameters(), lr=lr)
    trained_ce, val_acc_ce = train_model(model_ce, train_loader, val_loader, nn.CrossEntropyLoss(), opt_ce, epochs)
    test_acc_ce = evaluate_model(trained_ce, test_loader)

    # OVA
    print("\n=== OVA Loss ===")
    model_ova = YourCNN_OVA(best_feature_maps)
    opt_ova = optim.Adam(model_ova.parameters(), lr=lr)
    trained_ova, val_acc_ova = train_model(model_ova, train_loader, val_loader, OVALoss(), opt_ova, epochs, use_sigmoid=True)
    test_acc_ova = evaluate_model(trained_ova, test_loader, use_sigmoid=True)

    print("\n=== Summary ===")
    print(f"CrossEntropy -> Val Acc: {val_acc_ce:.4f}, Test Acc: {test_acc_ce:.4f}")
    print(f"OVA Loss     -> Val Acc: {val_acc_ova:.4f}, Test Acc: {test_acc_ova:.4f}")

    return {"CrossEntropy": (val_acc_ce, test_acc_ce),
            "OVA": (val_acc_ova, test_acc_ova)}


results = compare_ova_vs_crossentropy()


<span style="color:red">**Question 3.6:**</span> **Attack your best obtained model with PGD attacks with $\epsilon= 0.0313, k=20, \eta= 0.002$ on the testing set. Write the code for the attacks and report the robust accuracies. Also choose a random set of 20 clean images in the testing set and visualize the original and attacked images.**
   
<div style="text-align: right"> <span style="color:red">[4 points]</span> </div>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader


# PGD Attack

def pgd_attack(model, images, labels, eps=0.0313, alpha=0.002, iters=20, device="cpu"):
    model.eval()
    images, labels = images.to(device), labels.to(device)

    delta = torch.zeros_like(images, requires_grad=True)
    delta.data.uniform_(-eps, eps)

    for _ in range(iters):
        if delta.grad is not None:
            delta.grad.zero_()
        adv = (images + delta).clamp(0, 1)
        outputs = model(adv)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        delta.data = (delta + alpha * delta.grad.sign()).clamp(-eps, eps)
        delta.data = ((images + delta).clamp(0, 1) - images)

    return (images + delta).clamp(0, 1).detach()


# Robustness Evaluation

def evaluate_robustness(model, test_loader, eps=0.0313, alpha=0.002, iters=20, device="cpu"):
    model.eval()
    model.to(device)

    total, correct_clean, correct_adv = 0, 0, 0

    # Clean accuracy
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, pred = outputs.max(1)
            correct_clean += (pred == labels).sum().item()
            total += labels.size(0)

    clean_acc = 100. * correct_clean / total

    # Adversarial accuracy
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        adv = pgd_attack(model, images, labels, eps, alpha, iters, device)
        with torch.no_grad():
            outputs = model(adv)
            _, pred = outputs.max(1)
            correct_adv += (pred == labels).sum().item()

    adv_acc = 100. * correct_adv / total
    drop = clean_acc - adv_acc

    print(f"\nRobustness Results:")
    print(f"Clean Acc = {clean_acc:.2f}%")
    print(f"Adv Acc   = {adv_acc:.2f}%")
    print(f"Drop      = {drop:.2f}%")

    return clean_acc, adv_acc, drop


# Visualization

def visualize_adversarial_examples(model, test_dataset, num_samples=10,
                                   eps=0.0313, alpha=0.002, iters=20, device="cpu"):
    model.eval()
    model.to(device)

    np.random.seed(0)
    idxs = np.random.choice(len(test_dataset), num_samples, replace=False)

    fig, axes = plt.subplots(2, num_samples, figsize=(2*num_samples, 5))
    fig.suptitle(f"PGD Attack Visualization (ε={eps}, α={alpha}, k={iters})", fontsize=16)

    for i, idx in enumerate(idxs):
        img, label = test_dataset[idx]
        img_b = img.unsqueeze(0).to(device)
        label_b = torch.tensor([label], device=device)

        adv_b = pgd_attack(model, img_b, label_b, eps, alpha, iters, device)

        def to_numpy(t):
            return np.clip(t.squeeze(0).permute(1, 2, 0).cpu().numpy(), 0, 1)

        clean_np = to_numpy(img_b)
        adv_np   = to_numpy(adv_b)

        axes[0, i].imshow(clean_np)
        axes[0, i].set_title(f"Clean\nC:{label}")
        axes[0, i].axis("off")

        axes[1, i].imshow(adv_np)
        axes[1, i].set_title(f"Adv")
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()


# Run Attack

def run_attack(best_model, test_dataset, test_loader, device="cpu"):
    print(f"Using device: {device}")
    eps, alpha, iters = 0.0313, 0.002, 20

    clean_acc, adv_acc, drop = evaluate_robustness(
        best_model, test_loader, eps=eps, alpha=alpha, iters=iters, device=device
    )

    visualize_adversarial_examples(best_model, test_dataset, num_samples=10,
                                   eps=eps, alpha=alpha, iters=iters, device=device)

    return clean_acc, adv_acc, drop


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model = YourCNN([16,32,64], drop_rate=0.2, batch_norm=True, use_skip=True).to(device)

best_model.load_state_dict(torch.load("best_model.pth", map_location=device))

clean_acc, adv_acc, drop = run_attack(best_model, test_dataset, test_loader, device)


<span style="color:red">**Question 3.7:**</span> **Train a robust model using adversarial training with PGD ${\epsilon= 0.0313, k=10, \eta= 0.002}$. Write the code for the adversarial training and report the robust accuracies. After finishing the training, you need to store your best robust model in the folder `./models` and load the model to evaluate the robust accuracies for PGD and FGSM attacks with $\epsilon= 0.0313, k=20, \eta= 0.002$ on the testing set.**

**This task might take a long training time. Therefore, you can start from a pretrained model from the previous questions. Subsequently, you can apply adversarial training to train the robust model in a few epochs (e.g., 5 epochs). You can decide the *number of epochs* depending on your own situations. Our marking mainly bases on the correctness of your implementation.**  
   
<div style="text-align: right"> <span style="color:red">[4 points]</span> </div>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
import copy
import numpy as np
import matplotlib.pyplot as plt

def pgd_attack_training(model, images, labels, eps=0.0313, alpha=0.002, iters=10, device='cuda'):
    """
    PGD attack for adversarial training (lighter version with fewer iterations)

    Args:
        model: Target model
        images: Clean images [batch_size, 3, H, W]
        labels: True labels [batch_size]
        eps: Maximum perturbation (ε = 0.0313)
        alpha: Step size (η = 0.002)
        iters: Number of iterations (k = 10 for training)
        device: Device to run on

    Returns:
        adversarial_images: Perturbed images for training
    """
    model.eval()  # Set to eval mode for attack generation

    # Move to device
    images = images.to(device)
    labels = labels.to(device)

    # Initialize perturbation
    delta = torch.zeros_like(images, requires_grad=True)

    # Random initialization within eps ball
    delta.data.uniform_(-eps, eps)
    delta.data = torch.clamp(delta.data, 0.0 - images, 1.0 - images)

    criterion = nn.CrossEntropyLoss()

    # PGD iterations
    for i in range(iters):
        if delta.grad is not None:
            delta.grad.zero_()

        # Forward pass
        adv_images = images + delta
        outputs = model(adv_images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Update delta (gradient ascent)
        grad = delta.grad.detach()
        delta.data = delta.data + alpha * grad.sign()

        # Project back to eps ball and valid image range
        delta.data = torch.clamp(delta.data, -eps, eps)
        delta.data = torch.clamp(delta.data, 0.0 - images, 1.0 - images)

    # Generate final adversarial examples
    adversarial_images = torch.clamp(images + delta.detach(), 0.0, 1.0)

    return adversarial_images

def adversarial_training(model, train_loader, val_loader, criterion, optimizer,
                        num_epochs=10, eps=0.0313, alpha=0.002, iters=10,
                        adv_ratio=0.5, verbose=True, save_path=None):
    """
    Train model with adversarial training

    Args:
        model: Model to train
        train_loader: Training data loader
        val_loader: Validation data loader
        criterion: Loss function
        optimizer: Optimizer
        num_epochs: Number of training epochs
        eps: PGD epsilon for training (0.0313)
        alpha: PGD step size for training (0.002)
        iters: PGD iterations for training (10)
        adv_ratio: Ratio of adversarial examples in each batch
        save_path: Path to save the best model

    Returns:
        trained_model, best_robust_acc, training_history
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    best_robust_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses = []
    val_losses = []
    clean_val_accs = []
    robust_val_accs = []

    print(f"Starting Adversarial Training")
    print(f"Parameters: ε={eps}, η={alpha}, k={iters}, adv_ratio={adv_ratio}")
    print("=" * 60)

    for epoch in range(num_epochs):
        if verbose:
            print(f'Epoch {epoch+1}/{num_epochs}')
            print('-' * 40)

        # Training phase with adversarial examples
        model.train()
        running_loss = 0.0
        total_samples = 0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            batch_size = inputs.size(0)

            # Determine number of adversarial examples in this batch
            num_adv = int(batch_size * adv_ratio)
            num_clean = batch_size - num_adv

            # Split batch into clean and adversarial parts
            if num_adv > 0:
                # Generate adversarial examples for part of the batch
                adv_inputs = pgd_attack_training(model, inputs[:num_adv], labels[:num_adv],
                                               eps, alpha, iters, device)

                # Combine clean and adversarial examples
                if num_clean > 0:
                    combined_inputs = torch.cat([inputs[num_adv:], adv_inputs], dim=0)
                    combined_labels = torch.cat([labels[num_adv:], labels[:num_adv]], dim=0)
                else:
                    combined_inputs = adv_inputs
                    combined_labels = labels[:num_adv]
            else:
                combined_inputs = inputs
                combined_labels = labels

            # Training step
            model.train()  # Ensure training mode
            optimizer.zero_grad()
            outputs = model(combined_inputs)
            loss = criterion(outputs, combined_labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * batch_size
            total_samples += batch_size

            if verbose and batch_idx % 50 == 0:
                print(f'Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')

        train_loss = running_loss / total_samples

        # Validation phase - evaluate both clean and robust accuracy
        clean_acc, robust_acc, val_loss = evaluate_adversarial_model(
            model, val_loader, eps, alpha, iters, device, verbose=False
        )

        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        clean_val_accs.append(clean_acc)
        robust_val_accs.append(robust_acc)

        if verbose:
            print(f'Train Loss: {train_loss:.4f}')
            print(f'Val Clean Acc: {clean_acc:.4f}')
            print(f'Val Robust Acc: {robust_acc:.4f}')
            print(f'Val Loss: {val_loss:.4f}')
            print()

        # Save best model based on robust accuracy
        if robust_acc > best_robust_acc:
            best_robust_acc = robust_acc
            best_model_wts = copy.deepcopy(model.state_dict())

            # Save model if path provided
            if save_path:
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'clean_acc': clean_acc,
                    'robust_acc': robust_acc,
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                }, save_path)
                print(f"✓ Saved new best model with robust accuracy: {robust_acc:.4f}")

    # Load best model weights
    model.load_state_dict(best_model_wts)

    training_history = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'clean_val_accs': clean_val_accs,
        'robust_val_accs': robust_val_accs
    }

    return model, best_robust_acc, training_history

def evaluate_adversarial_model(model, test_loader, eps=0.0313, alpha=0.002, iters=20, device='cuda', verbose=True):
    """
    Evaluate model on both clean and adversarial examples

    Args:
        model: Model to evaluate
        test_loader: Test data loader
        eps: PGD epsilon for evaluation (0.0313)
        alpha: PGD step size for evaluation (0.002)
        iters: PGD iterations for evaluation (20)
        device: Device to run on
        verbose: Whether to print results

    Returns:
        clean_accuracy, robust_accuracy, avg_loss
    """
    model.eval()
    model = model.to(device)

    total_samples = 0
    correct_clean = 0
    correct_robust = 0
    total_loss = 0.0

    criterion = nn.CrossEntropyLoss()

    if verbose:
        print(f"Evaluating model robustness...")
        print(f"Attack parameters: ε={eps}, η={alpha}, k={iters}")

    with torch.no_grad():
        # First pass: Clean accuracy
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, predicted = torch.max(outputs.data, 1)

            total_samples += labels.size(0)
            correct_clean += (predicted == labels).sum().item()
            total_loss += loss.item() * labels.size(0)

    clean_accuracy = correct_clean / total_samples
    avg_loss = total_loss / total_samples

    # Second pass: Robust accuracy with stronger PGD attack
    correct_robust = 0
    batch_count = 0

    for images, labels in test_loader:
        batch_count += 1
        if verbose and batch_count % 20 == 0:
            print(f"Processing batch {batch_count}/{len(test_loader)}")

        images, labels = images.to(device), labels.to(device)

        # Generate adversarial examples with stronger attack (k=20)
        adversarial_images = pgd_attack_training(model, images, labels, eps, alpha, iters, device)

        # Evaluate on adversarial examples
        with torch.no_grad():
            outputs = model(adversarial_images)
            _, predicted = torch.max(outputs.data, 1)
            correct_robust += (predicted == labels).sum().item()

    robust_accuracy = correct_robust / total_samples

    if verbose:
        print(f"\nEvaluation Results:")
        print(f"Clean Accuracy:   {clean_accuracy:.4f} ({correct_clean}/{total_samples})")
        print(f"Robust Accuracy:  {robust_accuracy:.4f} ({correct_robust}/{total_samples})")
        print(f"Robustness Drop:  {(clean_accuracy - robust_accuracy):.4f}")
        print(f"Average Loss:     {avg_loss:.4f}")

    return clean_accuracy, robust_accuracy, avg_loss

def train_robust_model():
    """
    Main function to train robust model with adversarial training
    """
    print("ADVERSARIAL TRAINING FOR ROBUST MODEL")
    print("=" * 60)

    # Create model (start from best architecture or pretrained model)
    robust_model = YourCNN(
        list_feature_maps=[16, 32, 64],  # Use best configuration from previous experiments
        drop_rate=0.2,
        batch_norm=True,
        use_skip=True
    )

    # Optional: Load pretrained weights from previous best model
    # try:
    #     pretrained_path = '/models/best_model_previous.pth'
    #     if os.path.exists(pretrained_path):
    #         checkpoint = torch.load(pretrained_path)
    #         robust_model.load_state_dict(checkpoint['model_state_dict'])
    #         print("✓ Loaded pretrained model weights")
    # except:
    #     print("Starting from random initialization")

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Training setup
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(robust_model.parameters(), lr=0.001)

    # Adversarial training parameters as specified
    train_eps = 0.0313    # ε for training
    train_alpha = 0.002   # η for training
    train_iters = 10      # k for training

    # Model save path
    model_save_path = '/models/best_robust_model.pth'

    # Start adversarial training
    print("Starting adversarial training with parameters:")
    print(f"  Training ε: {train_eps}")
    print(f"  Training η: {train_alpha}")
    print(f"  Training k: {train_iters}")
    print(f"  Adversarial ratio: 50%")
    print()

    trained_robust_model, best_robust_acc, history = adversarial_training(
        model=robust_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=15,  # Can adjust based on your computational budget
        eps=train_eps,
        alpha=train_alpha,
        iters=train_iters,
        adv_ratio=0.5,  # 50% adversarial examples
        verbose=True,
        save_path=model_save_path
    )

    print(f"\n{'='*60}")
    print("ADVERSARIAL TRAINING COMPLETED")
    print(f"{'='*60}")
    print(f"Best robust validation accuracy: {best_robust_acc:.4f}")
    print(f"Model saved to: {model_save_path}")

    return trained_robust_model, history, model_save_path

def evaluate_robust_model(model_path):
    """
    Load and evaluate the trained robust model
    """
    print(f"\n{'='*60}")
    print("EVALUATING ROBUST MODEL")
    print(f"{'='*60}")

    # Load the trained robust model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Create model architecture
    robust_model = YourCNN(
        list_feature_maps=[16, 32, 64],
        drop_rate=0.2,
        batch_norm=True,
        use_skip=True
    )

    # Load trained weights
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)
        robust_model.load_state_dict(checkpoint['model_state_dict'])
        print(f"✓ Loaded robust model from {model_path}")
        print(f"  Training epoch: {checkpoint['epoch']}")
        print(f"  Training robust accuracy: {checkpoint['robust_acc']:.4f}")
    else:
        print(f"❌ Model file not found: {model_path}")
        return None

    # Create test loader
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Evaluation parameters (stronger attack as specified)
    eval_eps = 0.0313     # ε for evaluation
    eval_alpha = 0.002    # η for evaluation
    eval_iters = 20       # k for evaluation (stronger attack)

    print(f"\nEvaluating with stronger PGD attack:")
    print(f"  Evaluation ε: {eval_eps}")
    print(f"  Evaluation η: {eval_alpha}")
    print(f"  Evaluation k: {eval_iters}")

    # Comprehensive evaluation
    clean_acc, robust_acc, avg_loss = evaluate_adversarial_model(
        robust_model, test_loader, eval_eps, eval_alpha, eval_iters, device, verbose=True
    )

    return {
        'model': robust_model,
        'clean_accuracy': clean_acc,
        'robust_accuracy': robust_acc,
        'robustness_drop': clean_acc - robust_acc,
        'avg_loss': avg_loss
    }

def compare_with_baseline():
    """
    Compare robust model with baseline model
    """
    print(f"\n{'='*60}")
    print("COMPARISON: ROBUST vs BASELINE MODEL")
    print(f"{'='*60}")

    # Load test data
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Evaluation parameters
    eps = 0.0313
    alpha = 0.002
    iters = 20

    # Create and evaluate baseline model (assume you have this)
    baseline_model = YourCNN(
        list_feature_maps=[16, 32, 64],
        drop_rate=0.2,
        batch_norm=True,
        use_skip=True
    )

    # Load baseline weights if available
    # baseline_model.load_state_dict(torch.load('path_to_baseline_model.pth'))

    print("Evaluating Baseline Model...")
    baseline_clean, baseline_robust, _ = evaluate_adversarial_model(
        baseline_model, test_loader, eps, alpha, iters, device, verbose=False
    )

    # Load robust model results (from previous evaluation)
    robust_results = evaluate_robust_model('/models/best_robust_model.pth')

    if robust_results:
        robust_clean = robust_results['clean_accuracy']
        robust_robust = robust_results['robust_accuracy']

        print(f"\n{'='*50}")
        print("COMPARISON RESULTS")
        print(f"{'='*50}")
        print(f"{'Model':<15} | {'Clean Acc':<10} | {'Robust Acc':<11} | {'Drop':<6}")
        print("-" * 50)
        print(f"{'Baseline':<15} | {baseline_clean:.4f}     | {baseline_robust:.4f}      | {(baseline_clean-baseline_robust):.3f}")
        print(f"{'Robust (AT)':<15} | {robust_clean:.4f}     | {robust_robust:.4f}      | {(robust_clean-robust_robust):.3f}")

        print(f"\nKey Observations:")
        if robust_robust > baseline_robust:
            improvement = ((robust_robust - baseline_robust) / baseline_robust) * 100
            print(f"✅ Adversarial training IMPROVED robust accuracy by {improvement:.1f}%")
        else:
            decline = ((baseline_robust - robust_robust) / baseline_robust) * 100
            print(f"❌ Robust accuracy declined by {decline:.1f}%")

        if robust_clean < baseline_clean:
            clean_drop = ((baseline_clean - robust_clean) / baseline_clean) * 100
            print(f"⚠️  Clean accuracy decreased by {clean_drop:.1f}% (expected trade-off)")
        else:
            print(f"✅ Clean accuracy maintained or improved")

        print(f"\nRobustness Analysis:")
        print(f"- Baseline robustness drop: {(baseline_clean-baseline_robust):.4f}")
        print(f"- Robust model drop: {(robust_clean-robust_robust):.4f}")
        print(f"- Robustness improvement: {(baseline_clean-baseline_robust)-(robust_clean-robust_robust):.4f}")

# Execute the complete adversarial training pipeline
if __name__ == "__main__":
    # Step 1: Train robust model with adversarial training
    print("STEP 1: Training Robust Model")
    robust_model, training_history, model_path = train_robust_model()

    # Step 2: Evaluate robust model with stronger attacks
    print("\nSTEP 2: Evaluating Robust Model")
    evaluation_results = evaluate_robust_model(model_path)

    # Step 3: Compare with baseline
    print("\nSTEP 3: Comparing with Baseline")
    compare_with_baseline()

    # Step 4: Plot training curves
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.plot(training_history['train_losses'], label='Train Loss')
    plt.plot(training_history['val_losses'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 2)
    plt.plot(training_history['clean_val_accs'], label='Clean Accuracy')
    plt.plot(training_history['robust_val_accs'], label='Robust Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Clean vs Robust Validation Accuracy')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 3)
    clean_acc = evaluation_results['clean_accuracy'] if evaluation_results else 0
    robust_acc = evaluation_results['robust_accuracy'] if evaluation_results else 0

    plt.bar(['Clean Accuracy', 'Robust Accuracy'], [clean_acc, robust_acc],
            color=['blue', 'red'], alpha=0.7)
    plt.ylabel('Accuracy')
    plt.title('Final Test Results')
    plt.ylim(0, 1)

    for i, v in enumerate([clean_acc, robust_acc]):
        plt.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    print(f"\n{'='*60}")
    print("ADVERSARIAL TRAINING PIPELINE COMPLETED")
    print(f"{'='*60}")
    if evaluation_results:
        print(f"Final Test Results:")
        print(f"  Clean Accuracy: {evaluation_results['clean_accuracy']:.4f}")
        print(f"  Robust Accuracy: {evaluation_results['robust_accuracy']:.4f}")
        print(f"  Robustness Drop: {evaluation_results['robustness_drop']:.4f}")
    print(f"Model saved to: {model_path}")

<span style="color:red">**Question 3.8 (Kaggle competition)**</span>
<div style="text-align: right"> <span style="color:red">[10 points]</span> </div10

You can reuse the best model obtained in this assignment or develop new models to evaluate on the **testing set of the assignment Kaggle competion**. However, to gain any points for this question, your testing accuracy must **exceed** the accuracy threshold from a base model developed by us as shown in the leader board of the competition.

The marks for this question are as follows:
- If you are in *top 10%* of your cohort, you gain *10 points*.
- If you are in *top 20%* of your cohort, you gain *8 points*.
- If you *beat* the second boss model, you gain *6 points*.
- If you *beat* the first boss model, you gain *4 points*.

Moreover, if you rank in the top 10 of the entire cohort, you will receive a prestigious certificate signed by a renowned professor from our Faculty of Information Technology — a valuable highlight for your CV.
And for the overall winner, we have a special prize: *a book of your choice from Amazon, worth up to 150 AUDs*.